<a href="https://colab.research.google.com/github/erru-2005/Advance_profile_checker/blob/main/demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LightGlue Demo
In this notebook we match two pairs of images using LightGlue with early stopping and point pruning.

In [1]:
# If we are on colab: this clones the repo and installs the dependencies
from pathlib import Path

if Path.cwd().name != "LightGlue":
    !git clone --quiet https://github.com/cvg/LightGlue/
    %cd LightGlue
    !pip install --progress-bar off --quiet -e .

from lightglue import LightGlue, ALIKED
from lightglue.utils import load_image, rbd
from lightglue import viz2d
import torch

torch.set_grad_enabled(False)
images = Path("assets")

fatal: destination path 'LightGlue' already exists and is not an empty directory.
/content/LightGlue
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for lightglue (pyproject.toml) ... done


## Load extractor and matcher module
In this example we use SuperPoint features combined with LightGlue.

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

extractor = ALIKED(max_num_keypoints=2048).eval().to(device)

matcher = LightGlue(features="aliked").eval().to(device)

Downloading: "https://github.com/Shiaoming/ALIKED/raw/main/models/aliked-n16.pth" to /root/.cache/torch/hub/checkpoints/aliked-n16.pth


100%|██████████| 2.61M/2.61M [00:00<00:00, 12.9MB/s]


Downloading: "https://github.com/cvg/LightGlue/releases/download/v0.1_arxiv/aliked_lightglue.pth" to /root/.cache/torch/hub/checkpoints/aliked_lightglue_v0-1_arxiv.pth


100%|██████████| 45.4M/45.4M [00:01<00:00, 36.4MB/s]


## Easy example
The top image shows the matches, while the bottom image shows the point pruning across layers. In this case, LightGlue prunes a few points with occlusions, but is able to stop the context aggregation after 4/9 layers.

In [2]:
import time
import torch
import cv2
import numpy as np

from lightglue.utils import load_image, rbd
from lightglue import viz2d


# ============================================================
# 1. LOAD IMAGES
# ============================================================

print("=" * 70)
print("ALIKED + LIGHTGLUE IMAGE MATCHING TEST")
print("=" * 70)

image0_path = images / "image003.jfif"
image1_path = images / "images001.jfif"

print("\n[1] IMAGE INPUT")
print("-" * 70)
print("Reference image :", image0_path)
print("Camera image    :", image1_path)

image0 = load_image(image0_path)
image1 = load_image(image1_path)

print("Image 0 tensor shape :", tuple(image0.shape))
print("Image 1 tensor shape :", tuple(image1.shape))

print("Image 0 dtype :", image0.dtype)
print("Image 1 dtype :", image1.dtype)

print("Device       :", device)


# ============================================================
# 2. ALIKED FEATURE EXTRACTION
# ============================================================

print("\n[2] ALIKED FEATURE EXTRACTION")
print("-" * 70)

start = time.perf_counter()

with torch.inference_mode():
    feats0 = extractor.extract(image0.to(device))
    feats1 = extractor.extract(image1.to(device))

extract_time = time.perf_counter() - start

print(f"Extraction time : {extract_time:.4f} seconds")

print("\nFeature keys:")
print("Image 0:", list(feats0.keys()))
print("Image 1:", list(feats1.keys()))


# ============================================================
# 3. FEATURE INFORMATION
# ============================================================

print("\n[3] ALIKED FEATURES")
print("-" * 70)

for name, feats in [
    ("IMAGE 0", feats0),
    ("IMAGE 1", feats1)
]:

    print(f"\n{name}")

    if "keypoints" in feats:
        print("Keypoints shape :", tuple(feats["keypoints"].shape))
        print("Keypoints count :", feats["keypoints"].shape[-2])

    if "descriptors" in feats:
        print("Descriptors shape :", tuple(feats["descriptors"].shape))

    if "scores" in feats:
        print("Scores shape :", tuple(feats["scores"].shape))


# ============================================================
# 4. LIGHTGLUE MATCHING
# ============================================================

print("\n[4] LIGHTGLUE MATCHING")
print("-" * 70)

start = time.perf_counter()

with torch.inference_mode():
    matches01 = matcher({
        "image0": feats0,
        "image1": feats1
    })

match_time = time.perf_counter() - start

print(f"Matching time : {match_time:.4f} seconds")

print("\nMatch output keys:")
print(list(matches01.keys()))


# ============================================================
# 5. REMOVE BATCH DIMENSION
# ============================================================

feats0, feats1, matches01 = [
    rbd(x) for x in [feats0, feats1, matches01]
]


# ============================================================
# 6. EXTRACT KEYPOINTS + MATCHES
# ============================================================

kpts0 = feats0["keypoints"]
kpts1 = feats1["keypoints"]
matches = matches01["matches"]

print("\n[5] MATCH RESULTS")
print("-" * 70)

print("Image 0 keypoints :", len(kpts0))
print("Image 1 keypoints :", len(kpts1))
print("LightGlue matches :", len(matches))


# ============================================================
# 7. MATCH CONFIDENCE INFORMATION
# ============================================================

if "scores" in matches01:
    match_scores = matches01["scores"]

    print("\nMatch score information:")
    print("Score shape :", tuple(match_scores.shape))

    if len(match_scores) > 0:
        print("Minimum score :", float(match_scores.min()))
        print("Maximum score :", float(match_scores.max()))
        print("Mean score    :", float(match_scores.mean()))
        print("Median score  :", float(match_scores.median()))


# ============================================================
# 8. LIGHTGLUE ADAPTIVE INFORMATION
# ============================================================

print("\n[6] LIGHTGLUE INTERNAL INFORMATION")
print("-" * 70)

if "stop" in matches01:
    stop_value = matches01["stop"]

    if torch.is_tensor(stop_value):
        if stop_value.numel() == 1:
            stop_value = stop_value.item()

    print("Stopped after layers :", stop_value)
else:
    print("Stop information not available")


if "prune0" in matches01:
    print("Prune0 shape :", tuple(matches01["prune0"].shape))

if "prune1" in matches01:
    print("Prune1 shape :", tuple(matches01["prune1"].shape))


# ============================================================
# 9. GET MATCHED KEYPOINT COORDINATES
# ============================================================

print("\n[7] MATCHED KEYPOINT COORDINATES")
print("-" * 70)

if len(matches) > 0:

    m_kpts0 = kpts0[matches[..., 0]]
    m_kpts1 = kpts1[matches[..., 1]]

    print("Matched points image 0 :", tuple(m_kpts0.shape))
    print("Matched points image 1 :", tuple(m_kpts1.shape))

else:

    m_kpts0 = None
    m_kpts1 = None

    print("NO MATCHES FOUND")


# ============================================================
# 10. GEOMETRIC VERIFICATION
# ============================================================

print("\n[8] GEOMETRIC VERIFICATION")
print("-" * 70)

H = None
mask = None

if len(matches) >= 4:

    pts0 = m_kpts0.detach().cpu().numpy().astype(np.float32)
    pts1 = m_kpts1.detach().cpu().numpy().astype(np.float32)

    start = time.perf_counter()

    H, mask = cv2.findHomography(
        pts0,
        pts1,
        cv2.USAC_MAGSAC,
        3.0
    )

    geometry_time = time.perf_counter() - start

    print(f"Homography time : {geometry_time:.4f} seconds")

    if H is not None and mask is not None:

        mask = mask.ravel().astype(bool)

        inliers = int(np.sum(mask))
        total_matches = len(matches)

        inlier_ratio = inliers / total_matches

        print("Homography       : SUCCESS")
        print("Total matches    :", total_matches)
        print("Geometric inliers:", inliers)
        print(f"Inlier ratio     : {inlier_ratio:.2%}")

        print("\nHomography matrix:")
        print(H)

    else:

        print("Homography       : FAILED")
        print("Could not find a valid geometric transformation.")

else:

    print("Not enough matches for homography.")
    print("Minimum required:", 4)
    print("Available       :", len(matches))


# ============================================================
# 11. REPROJECTION ERROR
# ============================================================

print("\n[9] REPROJECTION ERROR")
print("-" * 70)

if H is not None and mask is not None and np.sum(mask) >= 4:

    pts0 = m_kpts0.detach().cpu().numpy().astype(np.float32)
    pts1 = m_kpts1.detach().cpu().numpy().astype(np.float32)

    projected = cv2.perspectiveTransform(
        pts0.reshape(-1, 1, 2),
        H
    ).reshape(-1, 2)

    errors = np.linalg.norm(
        projected - pts1,
        axis=1
    )

    inlier_errors = errors[mask]

    print(f"Mean reprojection error   : {np.mean(inlier_errors):.3f} px")
    print(f"Median reprojection error : {np.median(inlier_errors):.3f} px")
    print(f"Maximum reprojection error : {np.max(inlier_errors):.3f} px")

else:

    print("Reprojection error unavailable.")


# ============================================================
# 12. MATCH SPATIAL DISTRIBUTION
# ============================================================

print("\n[10] MATCH SPATIAL DISTRIBUTION")
print("-" * 70)

if len(matches) > 0:

    pts = m_kpts0.detach().cpu().numpy()

    h0, w0 = image0.shape[-2:]

    x = pts[:, 0]
    y = pts[:, 1]

    print(f"Image 0 dimensions : {w0} x {h0}")

    print(
        "Match bounding box : "
        f"x={x.min():.1f} → {x.max():.1f}, "
        f"y={y.min():.1f} → {y.max():.1f}"
    )

    coverage_width = (x.max() - x.min()) / w0
    coverage_height = (y.max() - y.min()) / h0

    print(f"Horizontal coverage : {coverage_width:.2%}")
    print(f"Vertical coverage   : {coverage_height:.2%}")

else:

    print("No spatial distribution available.")


# ============================================================
# 13. PROJECT REFERENCE IMAGE CORNERS
# ============================================================

print("\n[11] PROJECTED REFERENCE IMAGE")
print("-" * 70)

if H is not None:

    h0, w0 = image0.shape[-2:]

    corners = np.float32([
        [0, 0],
        [w0 - 1, 0],
        [w0 - 1, h0 - 1],
        [0, h0 - 1]
    ]).reshape(-1, 1, 2)

    projected_corners = cv2.perspectiveTransform(
        corners,
        H
    )

    projected_corners = projected_corners.reshape(-1, 2)

    print("Projected corners:")

    print("Top-left     :", projected_corners[0])
    print("Top-right    :", projected_corners[1])
    print("Bottom-right :", projected_corners[2])
    print("Bottom-left  :", projected_corners[3])

else:

    print("Cannot project corners because homography is unavailable.")


# ============================================================
# 14. FINAL DECISION
# ============================================================

print("\n" + "=" * 70)
print("FINAL MATCH ANALYSIS")
print("=" * 70)

total_matches = len(matches)

if mask is not None:

    inliers = int(np.sum(mask))
    inlier_ratio = inliers / total_matches

else:

    inliers = 0
    inlier_ratio = 0.0


if total_matches < 4:

    decision = "NO MATCH"

elif H is None:

    decision = "NO MATCH"

elif inliers < 10:

    decision = "UNCERTAIN"

elif inlier_ratio < 0.50:

    decision = "UNCERTAIN"

else:

    decision = "GEOMETRIC MATCH"


print("Total LightGlue matches :", total_matches)
print("Geometric inliers       :", inliers)
print(f"Inlier ratio            : {inlier_ratio:.2%}")
print("Final decision          :", decision)


# ============================================================
# 15. PERFORMANCE
# ============================================================

print("\n[12] PERFORMANCE")
print("-" * 70)

print(f"ALIKED extraction : {extract_time:.4f} sec")
print(f"LightGlue matching: {match_time:.4f} sec")
print(f"Total CV time     : {extract_time + match_time:.4f} sec")

print("\n" + "=" * 70)
print("TEST COMPLETE")
print("=" * 70)


# ============================================================
# 16. VISUALIZATION
# ============================================================

print("\n[13] VISUALIZATION")
print("-" * 70)

axes = viz2d.plot_images([
    image0,
    image1
])

if len(matches) > 0:

    viz2d.plot_matches(
        m_kpts0,
        m_kpts1,
        color="lime",
        lw=0.2
    )

    if "stop" in matches01:

        stop_value = matches01["stop"]

        if torch.is_tensor(stop_value):
            if stop_value.numel() == 1:
                stop_value = stop_value.item()

        viz2d.add_text(
            0,
            f"Stop after {stop_value} layers",
            fs=20
        )

if "prune0" in matches01 and "prune1" in matches01:

    kpc0 = viz2d.cm_prune(matches01["prune0"])
    kpc1 = viz2d.cm_prune(matches01["prune1"])

    viz2d.plot_images([
        image0,
        image1
    ])

    viz2d.plot_keypoints(
        [kpts0, kpts1],
        colors=[kpc0, kpc1],
        ps=10
    )

ModuleNotFoundError: No module named 'lightglue'

## Difficult example
For pairs with significant viewpoint- and illumination changes, LightGlue can exclude a lot of points early in the matching process (red points), which significantly reduces the inference time.

In [1]:
image0 = load_image(images / "sacre_coeur1.jpg")
image1 = load_image(images / "sacre_coeur2.jpg")

feats0 = extractor.extract(image0.to(device))
feats1 = extractor.extract(image1.to(device))
matches01 = matcher({"image0": feats0, "image1": feats1})
feats0, feats1, matches01 = [
    rbd(x) for x in [feats0, feats1, matches01]
]  # remove batch dimension

kpts0, kpts1, matches = feats0["keypoints"], feats1["keypoints"], matches01["matches"]
m_kpts0, m_kpts1 = kpts0[matches[..., 0]], kpts1[matches[..., 1]]

axes = viz2d.plot_images([image0, image1])
viz2d.plot_matches(m_kpts0, m_kpts1, color="lime", lw=0.2)
viz2d.add_text(0, f'Stop after {matches01["stop"]} layers')

kpc0, kpc1 = viz2d.cm_prune(matches01["prune0"]), viz2d.cm_prune(matches01["prune1"])
viz2d.plot_images([image0, image1])
viz2d.plot_keypoints([kpts0, kpts1], colors=[kpc0, kpc1], ps=6)

NameError: name 'load_image' is not defined